In [18]:
!pip install albumentations

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 18.5 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.23.4
    Uninstalling pydantic_core-2.23.4:
      Successfully uninstalled pydantic_core-2.23.4
  Attempting uninstall: pydantic
    Found existing installation: pydantic 1.10.12
    Uninstalling pydantic-1.10.12:
      Successfully uninstalled pydantic-1.10.12
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
from PIL import Image
from torchvision import transforms
import os
import json
import torch
import random
import shutil
from collections import defaultdict
import cv2
import albumentations as A

In [20]:
output_path = r"./TT100k"
os.makedirs(output_path,exist_ok = True)

annotation_path = r"../tt100k_2021/annotations_all.json"
delete_frame= []
def ThongKeAnnotation(annotation_path, test_count,train_count,val_count,rare_class=[], popular_class=[]):   
    with open(annotation_path, "r") as f:
        annotations = json.load(f)
    exists_categories = {}
    print(len(annotations["imgs"]))
    # thống kê các class tồn tại và tần suất xuất hiện trong annotations 
    for img_id, img_info in annotations["imgs"].items():
        for obj in img_info["objects"]:
            if  img_info['path'].split('/')[0] == 'other':
                img_info['path'] = f"val/{img_id}.jpg"
            if obj["category"] in exists_categories:
                exists_categories[obj["category"]] +=1
            else:
                exists_categories[obj["category"]] =1
            if img_info['path'].split('/')[0] == 'test':
                if obj["category"] in test_count:
                    test_count[obj["category"]] += 1
                else:
                    test_count[obj["category"]] = 1
            elif img_info['path'].split('/')[0] == 'train':
                if obj["category"] in train_count:
                    train_count[obj["category"]] += 1
                else:
                    train_count[obj["category"]] = 1
            elif img_info['path'].split('/')[0] == 'val':
                if obj["category"] in val_count:
                    val_count[obj["category"]] += 1
                else:
                    val_count[obj["category"]] = 1
    modified_annotation_path = os.path.join(output_path, "annotations_modified.json")
    with open(modified_annotation_path, "w") as file:
        json.dump(annotations, file, indent=4)  
    # tìm những class xuất hiện ít
    for key,value in exists_categories.items():
        if value < 50:
            rare_classes.append(key)
        elif value > 1000:
            popular_class.append(key)

    #in kết quả
    
    exists_categories = dict(sorted(exists_categories.items(), key=lambda item: item[1]))
    print(exists_categories)
    print(len(exists_categories))
    # print(exists_categories.values())
    test_count = dict(sorted(test_count.items(), key=lambda item: item[1]))
    train_count = dict(sorted(train_count.items(), key=lambda item: item[1]))
    val_count = dict(sorted(val_count.items(), key=lambda item: item[1]))
    print()
    print("test: length",len(test_count),test_count)
    print("test",test_count.values())
    print()
    print("train: ",len(train_count),train_count)
    print("train",train_count.values())
    print()
    print("val: ",len(val_count),val_count)
    print("val",val_count.values())

In [21]:

test_count = {}
train_count = {}
val_count = {}
# lấy những class có tần suất < 10 để đưa vào tập train
rare_classes = []
popular_classes = []
ThongKeAnnotation(annotation_path,test_count,train_count,val_count,rare_classes, popular_classes)
annotation_path = r"./TT100K/annotations_modified.json"

10592
{'w28': 1, 'w48': 1, 'w5': 1, 'pr5': 1, 'w49': 1, 'pa18': 1, 'pa6': 1, 'w44': 1, 'w1': 1, 'w62': 1, 'pw4.2': 1, 'pw2.5': 1, 'pr10': 1, 'pa8': 1, 'p7': 1, 'w14': 1, 'pclr': 1, 'w56': 1, 'ph3.8': 1, 'pw3.5': 2, 'pm13': 2, 'w16': 2, 'p20': 2, 'pr100': 2, 'pm1.5': 2, 'w12': 2, 'ph5.5': 2, 'p21': 2, 'ph1.8': 2, 'ph3.2': 2, 'pl65': 2, 'ph2.1': 2, 'pr45': 2, 'pcs': 2, 'p24': 2, 'ph4.4': 2, 'pnlc': 2, 'w38': 3, 'w31': 3, 'w60': 3, 'ph2.9': 3, 'w50': 3, 'pa12': 3, 'p28': 3, 'pm2.5': 3, 'ph2.6': 3, 'pmblr': 3, 'w15': 3, 'pw4.5': 3, 'w35': 3, 'ph2.4': 3, 'i11': 3, 'w37': 4, 'ph5.3': 4, 'w66': 4, 'p4': 4, 'pw3': 4, 'i14': 5, 'pn-2': 5, 'pm35': 5, 'w10': 5, 'p15': 5, 'phcs': 5, 'ph4.3': 6, 'i15': 6, 'pctl': 6, 'pm46': 6, 'pc': 7, 'w8': 7, 'pm2': 7, 'pt': 7, 'i3': 7, 'i1': 8, 'w43': 8, 'w46': 8, 'pm40': 8, 'pl25': 8, 'phclr': 9, 'pm25': 9, 'pm8': 9, 'pmr': 9, 'w41': 9, 'ph3.5': 9, 'pw4': 9, 'pr80': 11, 'pm50': 11, 'p1n': 12, 'ph2': 12, 'w18': 13, 'pa10': 13, 'w26': 13, 'p8': 13, 'ph4.8': 13, '

In [22]:
print(rare_classes)
print(len(rare_classes))

['pcr', 'w28', 'pl35', 'pw3.2', 'w24', 'p2', 'pl110', 'pmb', 'p14', 'w42', 'p16', 'pdd', 'pr70', 'p17', 'pw3.5', 'pcd', 'i14', 'i12', 'wc', 'pm13', 'w48', 'ph3', 'pc', 'ph3.3', 'w5', 'pl10', 'pss', 'phclr', 'i13', 'i1', 'w8', 'pm25', 'ph2.2', 'w47', 'w38', 'il50', 'pr20', 'w43', 'w16', 'p20', 'w34', 'w31', 'pn-2', 'iz', 'p1n', 'ph4.3', 'pm8', 'pm15', 'w3', 'w60', 'p29', 'w18', 'pr80', 'pa10', 'ph4.2', 'pa13', 'il70', 'w26', 'pm50', 'ph2', 'pr100', 'pm5', 'pr50', 'pm2', 'w45', 'pr5', 'pmr', 'pm1.5', 'w12', 'ph2.9', 'w50', 'i15', 'il110', 'p8', 'w49', 'pt', 'pm35', 'pa12', 'w41', 'p28', 'ph3.5', 'pw4', 'pm2.5', 'ph2.8', 'w37', 'ph5.3', 'ph5.5', 'ph2.6', 'w10', 'ph4.8', 'pmblr', 'p21', 'w15', 'pctl', 'w66', 'w46', 'ph1.8', 'w20', 'pm40', 'pl25', 'pa18', 'pa6', 'pw4.5', 'p15', 'ph2.5', 'pm46', 'p4', 'w35', 'i3', 'ph3.2', 'w44', 'pw3', 'ph2.4', 'pl65', 'w1', 'w62', 'pw4.2', 'phcs', 'ph2.1', 'pr45', 'pcs', 'pw2.5', 'i11', 'pr10', 'pa8', 'p24', 'p7', 'w14', 'ph4.4', 'pnlc', 'pclr', 'w56', 'ph

In [23]:
print(popular_classes)
print(len(popular_classes))

['pl40', 'pn', 'i5', 'p11', 'pne', 'pl50']
6


In [24]:
test_missing_classes = [cls for cls in test_count if cls not in train_count and cls not in val_count]
print("Classes only in test:", test_missing_classes)
val_missing_classes = [cls for cls in val_count if cls not in train_count and cls not in test_count]
print("Classes only in val:", val_missing_classes)
train_missing_classes = [cls for cls in train_count if cls not in test_count and cls not in val_count]
print("Classes only in train:", train_missing_classes)

Classes only in test: ['w28', 'pr45', 'pclr', 'w56', 'ph3.8']
Classes only in val: ['w48', 'ph3.3', 'pm25', 'w31', 'w26', 'pr5', 'pm1.5', 'w50', 'w49', 'ph2.6', 'pm46', 'w44', 'pl65', 'w1', 'w62', 'p7', 'ph4.4']
Classes only in train: ['w5', 'p20', 'pr100', 'ph2.9', 'pt', 'pa12', 'p21', 'ph1.8', 'pa18', 'pa6', 'pw4.5', 'pw4.2', 'pcs', 'pw2.5', 'pr10', 'pa8', 'p24', 'w14']


In [25]:
delete_frame_only = []
class_in_frame_in_rare_test = {}
class_in_frame_in_rare_train = {}
class_in_frame_in_rare_val = {}
def CheckFrameWithClasses(annotation_path, test_missing_classes, train_missing_classes, val_missing_classes, rare_class, popular_class, delete_frame_only, class_in_frame_in_rare_test={},class_in_frame_in_rare_train={},class_in_frame_in_rare_val={}):    
    with open(annotation_path, "r") as f:
        annotations = json.load(f)
    ex_test = defaultdict(int)
    ex_val = defaultdict(int)
    ex_train = defaultdict(int)

    frames_only_test = []
    frames_only_val = []
    frames_only_train = []
    frames_only_test_and_val = []
    frames_in_rare_test = []
    frames_in_rare_train = []
    frames_in_rare_val = []
    frames_in_rare_and_popular_test = []
    frames_in_rare_and_popular_train = []
    frames_in_rare_and_popular_val = []
    # thống kê các class tồn tại và tần suất xuất hiện trong annotations 
    for img_id, img_info in annotations["imgs"].items():
        check = False
        if img_info['path'].split('/')[0] == 'test':
            for obj in img_info["objects"]:
                if obj["category"] in test_missing_classes:
                    ex_test[obj["category"]] += 1
                    if check: frames_only_test_and_val.append(img_id)
                    check = True
                elif obj["category"] in val_missing_classes:
                    if check: frames_only_test_and_val.append(img_id)
                    check = True
        elif img_info['path'].split('/')[0] == 'train':
            for obj in img_info["objects"]:
                if obj["category"] in train_missing_classes:
                    ex_train[obj["category"]] += 1
        elif img_info['path'].split('/')[0] == 'val':
            for obj in img_info["objects"]:
                if obj["category"] in val_missing_classes:
                    ex_val[obj["category"]] += 1
                    if check: frames_only_test_and_val.append(img_id)
                    check = True
                if obj["category"] in test_missing_classes:
                    if check: frames_only_test_and_val.append(img_id)
                    check = True
              
    for img_id, img_info in annotations["imgs"].items():
        check = False
        if img_info['path'].split('/')[0] == 'test':
            for obj in img_info["objects"]:
                if obj["category"] in test_missing_classes:
                    check = True   
                elif obj["category"] in train_missing_classes or val_missing_classes:
                    check = False
                    break
            if check: frames_only_test.append(img_id)
        elif img_info['path'].split('/')[0] == 'train':
            for obj in img_info["objects"]:
                if obj["category"] in train_missing_classes:
                    check = True
                elif obj["category"] in test_missing_classes or val_missing_classes:
                    check = False
                    break
            if check: frames_only_train.append(img_id)
        elif img_info['path'].split('/')[0] == 'val':
            for obj in img_info["objects"]:
                if obj["category"] in val_missing_classes:
                    check = True
                elif obj["category"] in test_missing_classes or val_missing_classes:
                    check = False
                    break
            if check: frames_only_val.append(img_id)

    for img_id, img_info in annotations["imgs"].items():
        check = False
        for obj in img_info["objects"]:
            if img_info['path'].split('/')[0] == 'test':
                if obj['category'] in test_missing_classes:
                    frames_only_test.append(img_id)
                    check = True
                    break
            elif img_info['path'].split('/')[0] == 'train':
                if obj['category'] in train_missing_classes:
                    frames_only_train.append(img_id)
                    check = True
                    break
            elif img_info['path'].split('/')[0] == 'val':
                if obj['category'] in val_missing_classes:
                    frames_only_val.append(img_id)
                    check = True
                    break
            if check: break
                
    for img_id, img_info in annotations["imgs"].items():
        check = False
        if img_info['path'].split('/')[0] == 'test':
            for obj in img_info["objects"]:
                if obj["category"] in rare_class:
                    check = True   
                elif obj["category"] in popular_class:
                    check = False
                    break
            if check: frames_in_rare_test.append(img_id)
        elif img_info['path'].split('/')[0] == 'train':
            for obj in img_info["objects"]:
                if obj["category"] in rare_class:
                    check = True   
                elif obj["category"] in popular_class:
                    check = False
                    break
            if check: frames_in_rare_train.append(img_id)
        elif img_info['path'].split('/')[0] == 'val':
            for obj in img_info["objects"]:
                if obj["category"] in rare_class:
                    check = True   
                elif obj["category"] in popular_class:
                    check = False
                    break
            if check: frames_in_rare_val.append(img_id)

    for img_id, img_info in annotations["imgs"].items():
        check = False
        if img_info['path'].split('/')[0] == 'test':
            for obj in img_info["objects"]:
                if obj["category"] in rare_class:
                    check = True   
                elif obj["category"] in popular_class:
                    break
            if check: frames_in_rare_and_popular_test.append(img_id)
        elif img_info['path'].split('/')[0] == 'train':
            for obj in img_info["objects"]:
                if obj["category"] in rare_class:
                    check = True   
                elif obj["category"] in popular_class:
                    break
            if check: frames_in_rare_and_popular_train.append(img_id)
        elif img_info['path'].split('/')[0] == 'val':
            for obj in img_info["objects"]:
                if obj["category"] in rare_class:
                    check = True   
                elif obj["category"] in popular_class:
                    break
            if check: frames_in_rare_and_popular_val.append(img_id)

    delete_frame_only = frames_only_test + frames_only_val + frames_only_train + frames_only_test_and_val

    for img_id, img_info in annotations["imgs"].items():
        min_value = 100
        max_value = -1
        if img_id in frames_in_rare_test and img_id not in frames_only_test:
            for obj in img_info['objects']:
                for cl, value in test_count.items():
                    if cl in obj['category']:
                        if value < min_value: min_value = value
                        elif value > max_value: max_value = value
            class_in_frame_in_rare_test[img_id] = (min_value, max_value)
        elif img_id in frames_in_rare_train and img_id not in frames_only_train:
            for obj in img_info['objects']:
                for cl, value in test_count.items():
                    if cl in obj['category']:
                        if value < min_value: min_value = value
                        elif value > max_value: max_value = value
            class_in_frame_in_rare_train[img_id] = (min_value, max_value)
        elif img_id in frames_in_rare_val and img_id not in frames_only_val:
            for obj in img_info['objects']:
                for cl, value in test_count.items():
                    if cl in obj['category']:
                        if value < min_value: min_value = value
                        elif value > max_value: max_value = value
            class_in_frame_in_rare_val[img_id] = (min_value, max_value)
    #in kết quả
    ex_test = dict(sorted(ex_test.items(), key=lambda item: item[1]))
    ex_train = dict(sorted(ex_train.items(), key=lambda item: item[1]))
    ex_val = dict(sorted(ex_val.items(), key=lambda item: item[1]))
    
    print("test: length",len(ex_test),ex_test)
    print("class only in test",ex_test.values())
    print()
    print("train: ",len(ex_train),ex_train)
    print("class only in train",ex_train.values())
    print()
    print("val: ",len(ex_val),ex_val)
    print("class only in val",ex_val.values())
    print()
    print("Frame có class chỉ có trong 1 tập ", delete_frame_only)
    print("số lượng ", len(delete_frame_only))
    print()
    print("Frame có class hiếm trong tập test ", class_in_frame_in_rare_test)
    print("số lượng ", len(class_in_frame_in_rare_test))
    print()
    print("Frame có class hiếm trong tập train ", class_in_frame_in_rare_train)
    print("số lượng ", len(class_in_frame_in_rare_train))
    print()
    print("Frame có class hiếm trong tập val ", class_in_frame_in_rare_val)
    print("số lượng ", len(class_in_frame_in_rare_val))
    print()
    print("Frame có class hiếm và phổ biến trong tập test ", frames_in_rare_and_popular_test)
    print("số lượng ", len(frames_in_rare_and_popular_test))
    print()
    print("Frame có class hiếm và phổ biến trong tập train ", frames_in_rare_and_popular_train)
    print("số lượng ", len(frames_in_rare_and_popular_train))
    print()
    print("Frame có class hiếm và phổ biến trong tập val ", frames_in_rare_and_popular_val)
    print("số lượng ", len(frames_in_rare_and_popular_val))
    return delete_frame_only
    

In [26]:

delete_frame_only = CheckFrameWithClasses(annotation_path, test_missing_classes, train_missing_classes, val_missing_classes, rare_classes, popular_classes, delete_frame_only,class_in_frame_in_rare_test,class_in_frame_in_rare_train,class_in_frame_in_rare_val)
print(delete_frame_only)

test: length 5 {'w28': 1, 'pclr': 1, 'w56': 1, 'ph3.8': 1, 'pr45': 2}
class only in test dict_values([1, 1, 1, 1, 2])

train:  18 {'w5': 1, 'pa18': 1, 'pa6': 1, 'pw4.2': 1, 'pw2.5': 1, 'pr10': 1, 'pa8': 1, 'w14': 1, 'p20': 2, 'pr100': 2, 'p21': 2, 'ph1.8': 2, 'pcs': 2, 'p24': 2, 'ph2.9': 3, 'pa12': 3, 'pw4.5': 3, 'pt': 7}
class only in train dict_values([1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 3, 3, 3, 7])

val:  17 {'w48': 1, 'pr5': 1, 'w49': 1, 'w44': 1, 'w1': 1, 'w62': 1, 'p7': 1, 'pm1.5': 2, 'pl65': 2, 'ph4.4': 2, 'w31': 3, 'w50': 3, 'ph2.6': 3, 'pm46': 6, 'pm25': 9, 'w26': 13, 'ph3.3': 24}
class only in val dict_values([1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 3, 3, 3, 6, 9, 13, 24])

Frame có class chỉ có trong 1 tập  ['40317', '30363', '634', '68066', '65423', '32717', '20132', '29831', '1201', '74671', '97513', '55266', '73344', '74913', '85843', '73073', '38859', '14089', '26075', '98638', '89395', '84764', '52488', '44366', '43271', '52147', '74641', '49751', '2543', '48297', '94559',

In [27]:
folder_path = r"../tt100k_2021"
out_path = r"./TT100k"
os.makedirs(out_path, exist_ok=True)
transform = A.Compose([
    # A.Rotate(limit=10, p=1.0),  # Xoay tối đa ±10 độ
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0, p=1.0)  # Tăng giảm độ sáng ngẫu nhiên
])
def duplicate_frame(annotations, img_id, bboxes, categories, img, count):
    new_data = {}
    for i in range(1,count):
        # Áp dụng augmentation
        augmented = transform(image=img)
        augmented_img = augmented["image"]
        if i < 2:
             # Tạo augmented_img_id kiểu số nguyên
            augmented_img_id = int(f"{img_id}{i:03d}")  # Ghép ID gốc với số thứ tự (e.g., 62627001)
            # Lưu ảnh mới
            augmented_img_path = os.path.join(out_path, 'test', f"{augmented_img_id}.jpg")
            cv2.imwrite(augmented_img_path, augmented_img)
            # Tạo annotation mới
            new_data[augmented_img_id] = {
                "path": os.path.join('test', f"{augmented_img_id}.jpg"),
                "id": augmented_img_id,
                "objects": [
                    {"bbox": {"xmin": bbox['xmin'], "ymin": bbox['ymin'], "xmax": bbox['xmax'], "ymax": bbox['ymax']}, "category": cat} 
                    for bbox, cat in zip(bboxes, categories)
                ]
            }
            # Cập nhật annotation
            annotations["imgs"].update(new_data)
        elif i < 5:
             # Tạo augmented_img_id kiểu số nguyên
            augmented_img_id = int(f"{img_id}{i:03d}")  # Ghép ID gốc với số thứ tự (e.g., 62627001)
            # Lưu ảnh mới
            augmented_img_path = os.path.join(out_path, 'val', f"{augmented_img_id}.jpg")
            cv2.imwrite(augmented_img_path, augmented_img)
            # Tạo annotation mới
            new_data[augmented_img_id] = {
                "path": os.path.join('val', f"{augmented_img_id}.jpg"),
                "id": augmented_img_id,
                "objects": [
                    {"bbox": {"xmin": bbox['xmin'], "ymin": bbox['ymin'], "xmax": bbox['xmax'], "ymax": bbox['ymax']}, "category": cat} 
                    for bbox, cat in zip(bboxes, categories)
                ]
            }
            # Cập nhật annotation
            annotations["imgs"].update(new_data)
        else: 
             # Tạo augmented_img_id kiểu số nguyên
            augmented_img_id = int(f"{img_id}{i:03d}")  # Ghép ID gốc với số thứ tự (e.g., 62627001)
            # Lưu ảnh mới
            augmented_img_path = os.path.join(out_path, 'train', f"{augmented_img_id}.jpg")
            cv2.imwrite(augmented_img_path, augmented_img)
            # Tạo annotation mới
            new_data[augmented_img_id] = {
                "path": os.path.join('train', f"{augmented_img_id}.jpg"),
                "id": augmented_img_id,
                "objects": [
                    {"bbox": {"xmin": bbox['xmin'], "ymin": bbox['ymin'], "xmax": bbox['xmax'], "ymax": bbox['ymax']}, "category": cat} 
                    for bbox, cat in zip(bboxes, categories)
                ]
            }
            # Cập nhật annotation
            annotations["imgs"].update(new_data)
def case_duplicate_frame(annotations, img_info, img_path, img_id, class_in_frame_in_rare):
    bboxes = [obj["bbox"] for obj in img_info["objects"]]
    categories = [obj["category"] for obj in img_info["objects"]]
    img = cv2.imread(img_path)
    if class_in_frame_in_rare[img_id][0] < 11:
        duplicate_frame(annotations, img_id, bboxes, categories, img, count=105)
    elif class_in_frame_in_rare[img_id][0] < 26:
        duplicate_frame(annotations, img_id, bboxes, categories, img, count=54)
    elif class_in_frame_in_rare[img_id][0] < 51:
        duplicate_frame(annotations, img_id, bboxes, categories, img, count=28)
def augmentation_frame(folder_path, annotation_path, class_in_frame_in_rare_test, class_in_frame_in_rare_train, class_in_frame_in_rare_val, delete_frame_only):
    with open(annotation_path, "r") as f:
        annotations = json.load(f)
    t=0
    for img_id, img_info in annotations["imgs"].items():
        print(img_id)
        t+=1
        if t == 3: break
    # Tạo bản sao dữ liệu gốc để duyệt
    original_imgs = annotations["imgs"].copy()
    for img_id, img_info in original_imgs.items():
        img_path = os.path.join(folder_path, img_info['path'])
        if img_info['path'].split('/')[0] == 'val':
            img_path = os.path.join(folder_path, f"other/{img_id}.jpg")
        if img_id not in delete_frame_only:
            with Image.open(img_path) as img:
                # Lưu ảnh hợp lệ vào folder
                out_data = os.path.join(out_path, img_info['path'])
                if img_info['path'].split('/')[0] == 'val':
                    out_data = os.path.join(out_path, f"val/{img_id}.jpg")
                os.makedirs(os.path.dirname(out_data), exist_ok=True)
                img.save(out_data)
        if img_id in class_in_frame_in_rare_test:
            case_duplicate_frame(annotations, img_info, img_path, img_id, class_in_frame_in_rare_test)
        elif img_id in class_in_frame_in_rare_train:
            case_duplicate_frame(annotations, img_info, img_path, img_id, class_in_frame_in_rare_train)
        elif img_id in class_in_frame_in_rare_val:
           case_duplicate_frame(annotations, img_info, img_path, img_id, class_in_frame_in_rare_val)
    modified_annotation_path = os.path.join(out_path, "annotations_modified.json")
    with open(modified_annotation_path, "w") as file:
        json.dump(annotations, file, indent=4)
    print("Success")
def modify_annotation(delete_frame_only, annotation_path, out_path):
    # Mở file annotation
    with open(annotation_path, "r") as f:
        annotations = json.load(f)
    # Tạo bản sao dữ liệu gốc để duyệt
    original_imgs = annotations["imgs"].copy()
    print(delete_frame_only)
    removed_count = 0
    for img_id, img_info in original_imgs.items():
        if img_id in delete_frame_only or os.path.dirname(img_info['path']) == 'other':
            del annotations["imgs"][img_id]
            removed_count += 1
    print(f"Đã loại bỏ {removed_count} ảnh khỏi annotations.")
            
    # Lưu lại file annotation đã cập nhật
    modified_annotation_path = os.path.join(out_path, "annotations_modified.json")
    with open(modified_annotation_path, "w") as file:
        json.dump(annotations, file, indent=4)

    print("File annotation đã được cập nhật và lưu lại.")

In [28]:

modify_annotation(delete_frame_only, annotation_path, out_path)
augmentation_frame(folder_path, annotation_path, class_in_frame_in_rare_test, class_in_frame_in_rare_train, class_in_frame_in_rare_val, delete_frame_only)

['40317', '30363', '634', '68066', '65423', '32717', '20132', '29831', '1201', '74671', '97513', '55266', '73344', '74913', '85843', '73073', '38859', '14089', '26075', '98638', '89395', '84764', '52488', '44366', '43271', '52147', '74641', '49751', '2543', '48297', '94559', '66351', '70907', '54175', '71530', '2440', '13151', '69059', '43252', '91579', '15873', '75772', '57261', '98346', '67760', '98363', '20529', '11842', '19379', '32032', '2364', '95532', '84206', '47438', '98203', '29010', '55927', '36031', '2549', '6765', '46372', '69274', '96672', '31833', '23207', '20132', '29831', '1201', '74671', '34837', '11350', '97513', '55266', '23473', '48069', '73344', '74913', '85843', '73073', '38859', '53810', '14089', '26075', '98638', '14220', '89395', '97512', '84764', '52488', '44366', '43271', '52147', '74641', '49751', '2543', '48297', '94559', '66351', '70907', '54175', '71530', '2440', '13151', '69059', '75562', '43252', '91579', '68744', '15873', '75772', '57261', '98346', '6

In [29]:
#XÁC ĐỊNH FRAME CÓ CLASS CHỈ CHỨA TRONG TRAIN HOẶC VAL HOẶC TEST - DONE
#XÁC ĐỊNH FRAME CHỨA TRONG CLASS HIẾM VÀ PHỔ BIẾN - DONE
#XÁC ĐỊNH FRAME CHỈ CHỨA CLASS HIẾM - DONE
#XỬ LÍ TRƯỜNG HỢP ONLY TRAIN, VAL, TEST
    #XÓA CÁC ẢNH ĐÓ
#SAU ĐÓ XỬ LÝ MẤY TRƯỜNG HỢP HIẾM 
    #AUGMENTATION: XOAY 10 ĐỘ, LÀM TẮNG, TỐI ĐỘ SÁNG
    #AUGMENTATION ẢNH CÓ CLASS HIẾM Ở TẬP TEST, TRAIN, VAL NHƯ SAU:
        #ĐỐI VỚI CLASS CHỈ XUẤT HIỆN 1-10 LẦN THÌ LÀM THÊM 100 ẢNH CHO TẬP TRAIN, 3 ẢNH CHO TẬP VAL, 1 ảnh cho test
        #ĐỐI VỚI CLASS CHỈ XUẤT HIỆN 11-25 LẦN THÌ LÀM THÊM 50 ẢNH CHO TẬP TRAIN, 2 ẢNH CHO TẬP VAL, 1 ẢNH CHO TEST
        #ĐỐI VỚI CLASS CHỈ XUẤT HIỆN 26-50 LẦN THÌ LÀM THÊM 25 ẢNH CHO TẬP TRAIN, 1 ẢNH CHO TẬP VAL, 1 ẢNH CHO TEST
#XỬ LÝ TRƯỜNG HỢP HIẾM + PHỔ BIẾN NẾU CÓ
        #XÉT SAU KHI THẤY THỐNG KÊ LẠI KO ỔN